In [28]:
.libPaths(c("/home/mattia/miniconda3/envs/Signac/lib/R/library","/home/mattia/R/x86_64-pc-linux-gnu-library/4.2"))
library(tidyverse)
library(IRanges)
library(dplyr)
library(plyranges)
library(ChIPpeakAnno)
library(tibble)
library(readr)
library(biomaRt)
library(plyranges)
library(parallel)
library(ggthemes)
library(viridis)
library(MASS)
library(ashr)
library(LSD)
library(plotfunctions)
library(colorRamps)
library(ggplot2)
library(GenomicAlignments)
library(GenomicFeatures)
library(Rsamtools)
library(parallel)
library(TxDb.Hsapiens.UCSC.hg38.knownGene)
library(org.Hs.eg.db)
setwd("/date/gcb/gcb_MZ/Round1/")

In [23]:
files <- list.files("/date/gcb/gcb_MZ/Chrom_HMM_hg38_annotation", pattern = "\\.bed", full.names = T)
encode <-  lapply(files, read_delim, delim="\t", skip = 1, col_names = F) 

names(encode) <- str_split_fixed(as.character(list.files("/date/gcb/gcb_MZ/Chrom_HMM_hg38_annotation", 
                                            pattern = "\\.bed", full.names = F)), pattern = "_", 5)[,1]


state <- c("1_TssA","2_PromU","3_PromD1","4_PromD2","5_Tx5'","6_Tx","7_Tx3'",
           "8_TxWk","9_TxReg","10_TxEnh5'","11_TxEnh3'","12_TxEnhW","13_EnhA1",
           "14_EnhA2","15_EnhAF","16_EnhW1","17_EnhW2","18_EnhAc","19_DNase",
           "20_ZNF/Rpts","21_Het","22_PromP","23_PromBiv","24_ReprPC","25_Quies")

TSS<-list()
for (i in 1:length(encode)) {
   join <- makeGRangesFromDataFrame((encode[[i]] %>% dplyr::rename(chr=1,start=2,end=3,annotation=4)),keep.extra.columns = T) %>% 
    as.data.frame()%>% 
    dplyr::filter(annotation==c("1_TssA"))
  TSS[[i]] <- join
}

Rows: 939198 Columns: 9
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (4): X1, X4, X6, X9
dbl (5): X2, X3, X5, X7, X8

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 844009 Columns: 9
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (4): X1, X4, X6, X9
dbl (5): X2, X3, X5, X7, X8

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 751511 Columns: 9
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (4): X1, X4, X6, X9
dbl (5): X2, X3, X5, X7, X8

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 781986 Col

In [29]:
TSS<- bind_rows(TSS)
#annotate the list of TSS
TSS_annotate <- ChIPseeker::annotatePeak(makeGRangesFromDataFrame(TSS,keep.extra.columns=TRUE),
                                                            tssRegion=c(-2500, 2500), TxDb=TxDb.Hsapiens.UCSC.hg38.knownGene::TxDb.Hsapiens.UCSC.hg38.knownGene, 
                                                            annoDb="org.Hs.eg.db") %>% 
  as_tibble() %>% 
  dplyr::rename(chr=seqnames,ensembl_gene_id=ENSEMBL) %>% 
  mutate(Feature=annotation,
         Feature=gsub(" \\(.*","",Feature),
         Feature=gsub("Distal Intergenic","Intergenic",Feature),
         Feature=gsub("3' UTR","Exon",Feature),
         Feature=gsub("5' UTR","Exon",Feature),
         Feature=gsub("Downstream","Intergenic",Feature)) %>% 
  replace(., is.na(.), "0")



Registered S3 methods overwritten by 'treeio':
  method              from    
  MRCA.phylo          tidytree
  MRCA.treedata       tidytree
  Nnode.treedata      tidytree
  Ntip.treedata       tidytree
  ancestor.phylo      tidytree
  ancestor.treedata   tidytree
  child.phylo         tidytree
  child.treedata      tidytree
  full_join.phylo     tidytree
  full_join.treedata  tidytree
  groupClade.phylo    tidytree
  groupClade.treedata tidytree
  groupOTU.phylo      tidytree
  groupOTU.treedata   tidytree
  inner_join.phylo    tidytree
  inner_join.treedata tidytree
  is.rooted.treedata  tidytree
  nodeid.phylo        tidytree
  nodeid.treedata     tidytree
  nodelab.phylo       tidytree
  nodelab.treedata    tidytree
  offspring.phylo     tidytree
  offspring.treedata  tidytree
  parent.phylo        tidytree
  parent.treedata     tidytree
  root.treedata       tidytree
  rootnode.phylo      tidytree
  sibling.phylo       tidytree



>> preparing features information...		 2023-11-13 04:26:32 PM 
>> identifying nearest features...		 2023-11-13 04:26:33 PM 
>> calculating distance from peak to TSS...	 2023-11-13 04:26:52 PM 
>> assigning genomic annotation...		 2023-11-13 04:26:52 PM 
>> adding gene annotation...			 2023-11-13 04:27:31 PM 


'select()' returned 1:many mapping between keys and columns



>> assigning chromosome lengths			 2023-11-13 04:27:33 PM 
>> done...					 2023-11-13 04:27:33 PM 


In [31]:
write_tsv(TSS_annotate,"TSS_chromHMM_annotate.bed")